# Efficient Action Recognition for Home Monitoring

First, install the Python Package that contains the source code that enables preprocessing the input videos and doing the action recognition inference

In [ ]:
#%pip install -e .

We import the required functions to run the inference on a sample input

In [ ]:
import numpy as np
from action_recognition_home_monitoring import RGBI3D, TwoStream

Firstly, we obtain the action list with the name and identifier that this solution is able to identify

In [ ]:
import json
with open('res/classes.json') as json_file:
    classes_to_id = json.load(json_file)
    id_to_classes = {v: k for k,v in classes_to_id.items()}
    actions_list = [k for k in classes_to_id.keys()]

### Video Action Recognition Model
Create the video model instance with a reduced spatial resolution (112x112) to analyze 64 frames videos
recorded at 25 FPS (which represents 2.56 seconds clips). We also load the model trained
weights to perform action recognition.

We can select which model to choose based on their temporal resolution. Bear in mind that is the model internally
the one that does the temporal sampling.

In [ ]:
video_model = RGBI3D(model_keras_file="./models_weights/RGBI3D_16_112.keras")
# video_model = RGBI3D(model_keras_file="./models_weights/RGBI3D_32_112.keras")
# video_model = RGBI3D(model_keras_file="./models_weights/RGBI3D_64_112.keras")

#### 1. Load and preprocess an input video to perform action recognition
Firstly, import the *load_video* function from the preprocessing module. Then provide a path to a video you want to
analyze. We obtain a list of clips of 64 frames each that represent the video.

In [ ]:
from action_recognition_home_monitoring import load_video
video_file_name = "action_fall_sample.mp4"
video_path = "./res/video/{}".format(video_file_name)
video_clips = load_video(video_path)

#### 2. Show the loaded video
Show the first clip of the video of a person walking. Note that **matplotlib** is required to plot the images

In [ ]:
from action_recognition_home_monitoring import show_video
show_video(video_clips[0], frames=5)

#### 3. Infer the activity using the optimized RGBI3D network
Firstly, we create a function to infer the data with the DL model easily

In [ ]:
def predict_data(model, video_path, actions_list):
    prediction = model.predict(video_path, batch_size=8)
    prediction = prediction.mean(axis=0)
    detected_action = np.argmax(prediction, axis=-1)

    print("Action recognizer detected: {} - with a confidence of: {:0.2f}%".format(actions_list[detected_action],
                                                                                   prediction[detected_action]*100))

We analyze the video with the Video network. Note how the most efficient video network identifies with a 47% confidence that someone is **falling down**

In [ ]:
predict_data(model=video_model, video_path=video_path, actions_list=actions_list)

### Two Stream Action Recognition Model
In addition, we can also do inference with our most accurate alternative, that is the **Two Stream** model. To use this model, we first compute the Optical Flow on input video, and then we do inference on the `RGBI3D_64_122` and `OpticalFlow_64_122` trained architectures.

This is straightforward to use, as we can rely on the `action_recognition_home_monitoring package`


In [ ]:
two_stream_model = TwoStream(rgb_model_keras_file="./models_weights/RGBI3D_64_112.keras",
                             flow_model_keras_file="./models_weights/OpticalFlow_64_112.keras")

Now, we do inference on the same input data as before and we observe the difference in confidence for recognizing the critical action falling down. In the case, confidence goes up for up to 99%.

In [ ]:
predict_data(model=two_stream_model, video_path=video_path, actions_list=actions_list)